# ML Model Training — Asthma Classification

Train, evaluate, and tune classifiers to predict asthma diagnosis from patient features.

### Опис Компанії-Замовника  
BreathWell Clinic — спеціалізований медичний заклад, що займається діагностикою, моніторингом та лікуванням захворювань дихальних шляхів, зокрема астми.

### Запит  
Клініка прагне покращити діагностичну точність, щоб виявляти пацієнтів з астмою, навіть за мінімальними ознаками, та запобігти прогресуванню хвороби.

### Пропозиція Реалізації  
Створити модель машинного навчання для класифікації ймовірності астми на основі анамнестичних, клініко-параклінічних даних і симптомів. Інструмент інтегрується в електронну медичну карту, допомагаючи лікарям виконати скринінг пацієнтів на ранніх етапах.

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
import joblib
import os
from datetime import datetime
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore')

DATABASE_CONFIG = {
    'host': 'localhost',
    'database': 'breathwell_clinic',
    'user': 'postgres',
    'password': '0702',
    'port': '5432'
}

connection_string = (
    f"postgresql://{DATABASE_CONFIG['user']}:{DATABASE_CONFIG['password']}"
    f"@{DATABASE_CONFIG['host']}:{DATABASE_CONFIG['port']}"
    f"/{DATABASE_CONFIG['database']}"
)

engine = create_engine(connection_string)

df = pd.read_sql_table('asthma_patients_data', engine)

print("Інформація про датасет:")
print(f"Розмір: {df.shape}")
print(f"Кількість записів: {df.shape[0]}")
print(f"Кількість ознак: {df.shape[1]}")
print(f"\nПерші 5 рядків:")
display(df.head())

os.makedirs('models', exist_ok=True)


Інформація про датасет:
Розмір: (2392, 34)
Кількість записів: 2392
Кількість ознак: 34

Перші 5 рядків:


,patientid,age,gender,bmi,smoking,physicalactivity,dietquality,sleepquality,pollutionexposure,pollenexposure,...,exerciseinduced,diagnosis,doctorincharge,timestamp,ethnicity_1,ethnicity_2,ethnicity_3,educationlevel_1,educationlevel_2,educationlevel_3
0,5034,0.965740,-0.986710,-1.582769,-0.406355,-1.432099,0.160113,0.971063,0.809355,-0.780866,...,0.808131,0,Dr_Confid,2025-04-10 23:35:08,2.008927,-0.325379,-0.320644,-0.799674,-0.675184,-0.327731
1,5035,-0.747054,1.013469,-0.623300,-0.406355,0.291269,0.453069,-1.076746,-1.036866,0.810184,...,0.808131,0,Dr_Confid,2025-08-30 18:23:59,-0.497778,3.073339,-0.320644,-0.799674,1.481078,-0.327731
2,5036,0.687989,-0.986710,-1.229074,-0.406355,0.581330,1.434458,-0.102976,-1.210374,-1.267434,...,0.808131,0,Dr_Confid,2025-01-07 21:12:22,-0.497778,3.073339,-0.320644,1.250509,-0.675184,-0.327731
3,5037,-0.098970,1.013469,1.565307,-0.406355,-1.256398,0.276233,-1.596880,-1.509757,0.849659,...,-1.237424,0,Dr_Confid,2025-11-01 18:22:40,-0.497778,3.073339,-0.320644,1.250509,-0.675184,-0.327731
4,5038,0.873156,-0.986710,-1.105686,-0.406355,-0.154081,-0.651625,1.504976,-1.373822,-0.713717,...,0.808131,0,Dr_Confid,2025-04-12 03:14:15,-0.497778,-0.325379,-0.320644,-0.799674,-0.675184,3.051286


In [2]:
try:
    with engine.connect() as conn:
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS predictions (
                id SERIAL PRIMARY KEY,
                patientid INTEGER,
                true_label INTEGER,
                predicted_label INTEGER,
                predicted_proba FLOAT,
                model_name VARCHAR(255),
                source VARCHAR(50),
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """))
        conn.commit()
    print("Таблиця predictions створена")
except Exception as e:
    print(f"Таблиця predictions вже існує або помилка: {e}")

try:
    with engine.connect() as conn:
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS model_metrics (
                id SERIAL PRIMARY KEY,
                model_name VARCHAR(255),
                dataset_type VARCHAR(50),
                accuracy FLOAT,
                precision FLOAT,
                recall FLOAT,
                f1_score FLOAT,
                roc_auc FLOAT,
                hyperparameters TEXT,
                optimized BOOLEAN DEFAULT FALSE,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """))
        conn.commit()
    print("Таблиця model_metrics створена")
except Exception as e:
    print(f"Таблиця model_metrics вже існує або помилка: {e}")


Таблиця predictions створена
Таблиця model_metrics створена


## Завдання 1: Вибір техніки моделювання

Мета така, що потрібно передбачити чи є у пацієнта астма чи нема. Це діагноз, який може бути або так або ні, тобто модель має віднести пацієнта до одного з цих двох класів. Лікарю потрібна чітка відповідь щоб прийняти рішення про лікування, тому вибираємо задачу класифікації.

In [3]:
print("Аналіз цільової змінної:")
print(f"\nРозподіл класів у цільовій змінній diagnosis:")
class_counts = df['diagnosis'].value_counts()
print(class_counts)
print(f"\nВідсотковий розподіл:")
print(df['diagnosis'].value_counts(normalize=True) * 100)

balance_ratio = class_counts.min() / class_counts.max()
print(f"\nБаланс класів): {balance_ratio:.3f}")
if balance_ratio < 0.5:
    print("Дані незбалансовані")
else:
    print("Дані досить збалансовані")


Аналіз цільової змінної:

Розподіл класів у цільовій змінній diagnosis:
diagnosis
0    2268
1     124
Name: count, dtype: int64

Відсотковий розподіл:
diagnosis
0    94.816054
1     5.183946
Name: proportion, dtype: float64

Баланс класів): 0.055
Дані незбалансовані


## Завдання 2: Опис модельних припущень

Logistic Regression припускає що зв'язок між ознаками та результатом лінійний.

Random Forest не потребує лінійності, може знайти нелінійні зв'язки. Стійкий до кореляцій між ознаками. Підходить для наших даних бо у нас багато різних ознак.

Gradient Boosting теж може працювати з нелінійними зв'язками та знаходити складні взаємодії між ознаками. Може знайти складні патерни в медичних даних.

Тому всі три моделі підходять для нашої задачі.


In [4]:
exclude_cols = ['patientid', 'timestamp', 'diagnosis', 'doctorincharge']
feature_cols = [col for col in df.columns if col not in exclude_cols]

X = df[feature_cols]
y = df['diagnosis'].astype(int)

print(f"Кількість ознак: {len(feature_cols)}")
print(f"Кількість спостережень: {len(X)}")
print(f"Відношення спостережень до ознак: {len(X) / len(feature_cols):.2f}")

print("\nПеревірка припущень:")
print("Дані вже масштабовані ")
print("Немає пропущених значень ")
print("Категоріальні ознаки закодовані ")

print("\nКореляції між основними ознаками (перші 10):")
corr_matrix = X.iloc[:, :10].corr()
high_corr = (corr_matrix.abs() > 0.8) & (corr_matrix.abs() < 1.0)
if high_corr.any().any():
    print("Знайдено високі кореляції (>0.8), але Random Forest та Gradient Boosting стійкі до цього")
else:
    print("Серйозних кореляцій не виявлено")

print("\nВисновок:")
print("Logistic Regression може працювати добре завдяки масштабуванню")
print("Random Forest та Gradient Boosting найбільш універсальні для нашої задачі")
print("Дані готові для моделювання")


Кількість ознак: 30
Кількість спостережень: 2392
Відношення спостережень до ознак: 79.73

Перевірка припущень:
Дані вже масштабовані 
Немає пропущених значень 
Категоріальні ознаки закодовані 

Кореляції між основними ознаками (перші 10):
Серйозних кореляцій не виявлено

Висновок:
Logistic Regression може працювати добре завдяки масштабуванню
Random Forest та Gradient Boosting найбільш універсальні для нашої задачі
Дані готові для моделювання


## Завдання 3: Вибір метрик для оцінки моделі

У медицині важливо правильно вибрати метрики бо помилки мають різну вагу. Якщо модель не знайде хворого пацієнта це гірше ніж якщо помилково помітить здорового.

Accuracy показує загальну частку правильних передбачень. Але при незбалансованих класах може бути не дуже правдивою.

Precision показує із усіх пацієнтів яких модель назвала хворими скільки справді хворі. Мінімізує false positives - коли ми кажемо що є астма а її немає.

Recall найважливіший для медицини. Показує із усіх справді хворих скільки модель знайшла. Мінімізує false negatives - коли є астма але ми її не виявили. Пропустити хворого це найгірше що може статися.

F1-Score це баланс між Precision та Recall. Добре для порівняння моделей.

ROC-AUC показує загальну якість моделі на різних порогах.

Для оптимізації використовуємо F1-Score бо вона балансує обидва аспекти. Але особливу увагу приділяємо Recall бо в медицині краще перевірити здорових ніж пропустити хворих.


In [5]:
def calculate_metrics(y_true, y_pred, y_proba=None, model_name="model", dataset_type="test"):
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1_score': f1_score(y_true, y_pred, zero_division=0)
    }
    
    if y_proba is not None:
        metrics['roc_auc'] = roc_auc_score(y_true, y_proba)
    else:
        metrics['roc_auc'] = None
    
    print(f"\nМетрики для {model_name} ({dataset_type}):")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1-Score:  {metrics['f1_score']:.4f}")
    if metrics['roc_auc'] is not None:
        print(f"  ROC-AUC:   {metrics['roc_auc']:.4f}")
    
    cm = confusion_matrix(y_true, y_pred)
    print(f"\nМатриця плутанини:")
    print(f"  True Negatives (TN):  {cm[0,0]}")
    print(f"  False Positives (FP): {cm[0,1]}")
    print(f"  False Negatives (FN): {cm[1,0]}")
    print(f"  True Positives (TP):  {cm[1,1]}")
    
    return metrics

print("Функція для обчислення метрик створена")
print("Відстежуватимемо: Accuracy, Precision, Recall, F1-Score, ROC-AUC")

Функція для обчислення метрик створена
Відстежуватимемо: Accuracy, Precision, Recall, F1-Score, ROC-AUC


## Завдання 4: Поділ даних на навчальні та тестові

Розділяємо дані на тренувальну та тестову вибірки. Використовуємо train_test_split з параметром stratify щоб зберегти пропорції класі (якщо в тренуванні буде більше одного класу а в тесті іншого то оцінка моделі буде неправильною)

Не використовуємо TimeSeriesSplit бо наші дані не мають часової залежності. Хоча є timestamp але порядок записів не важливий для задачі діагностики.

Поділяємо 80% на тренування та 20% на тест. На наших 2392 записах вийде близько 1913 на тренування та 479 на тест і цього достатньо для обох вибірок.


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Розмір тренувальної вибірки: {X_train.shape}")
print(f"Розмір тестової вибірки: {X_test.shape}")

print(f"\nРозподіл класів у тренувальній вибірці:")
train_dist = y_train.value_counts()
print(train_dist)
print(f"Пропорції: {y_train.value_counts(normalize=True) * 100}")

print(f"\nРозподіл класів у тестовій вибірці:")
test_dist = y_test.value_counts()
print(test_dist)
print(f"Пропорції: {y_test.value_counts(normalize=True) * 100}")

train_ratio = train_dist[1] / train_dist[0]
test_ratio = test_dist[1] / test_dist[0]
print(f"\nПеревірка збереження пропорцій:")
print(f"Відношення класів у тренуванні: {train_ratio:.3f}")
print(f"Відношення класів у тесті: {test_ratio:.3f}")
print(f"Різниця: {abs(train_ratio - test_ratio):.4f}")

if abs(train_ratio - test_ratio) < 0.01:
    print("Пропорції успішно збережені")
else:
    print("Пропорції трохи відрізняються але це прийнятно")


Розмір тренувальної вибірки: (1913, 30)
Розмір тестової вибірки: (479, 30)

Розподіл класів у тренувальній вибірці:
diagnosis
0    1814
1      99
Name: count, dtype: int64
Пропорції: diagnosis
0    94.824882
1     5.175118
Name: proportion, dtype: float64

Розподіл класів у тестовій вибірці:
diagnosis
0    454
1     25
Name: count, dtype: int64
Пропорції: diagnosis
0    94.780793
1     5.219207
Name: proportion, dtype: float64

Перевірка збереження пропорцій:
Відношення класів у тренуванні: 0.055
Відношення класів у тесті: 0.055
Різниця: 0.0005
Пропорції успішно збережені


## Завдання 5: Тренування моделей на train-даних

Тренуємо три моделі для порівняння: Logistic Regression, Random Forest та Gradient Boosting. Після тренування зберігаємо передбачення моделей у таблицю predictions з source="train".


In [7]:
def save_predictions_to_db(y_true, y_pred, y_proba, patient_ids, model_name, source):
    predictions_df = pd.DataFrame({
        'patientid': patient_ids,
        'true_label': y_true,
        'predicted_label': y_pred,
        'predicted_proba': y_proba,
        'model_name': model_name,
        'source': source
    })
    
    try:
        predictions_df.to_sql('predictions', engine, if_exists='append', index=False)
        print(f"Передбачення збережені в БД ({len(predictions_df)} записів)")
    except Exception as e:
        print(f"Помилка збереження: {e}")

def save_metrics_to_db(model_name, dataset_type, metrics, hyperparameters="{}", optimized=False):
    metrics_df = pd.DataFrame([{
        'model_name': model_name,
        'dataset_type': dataset_type,
        'accuracy': metrics['accuracy'],
        'precision': metrics['precision'],
        'recall': metrics['recall'],
        'f1_score': metrics['f1_score'],
        'roc_auc': metrics['roc_auc'],
        'hyperparameters': str(hyperparameters),
        'optimized': optimized
    }])
    
    try:
        metrics_df.to_sql('model_metrics', engine, if_exists='append', index=False)
        print(f"Метрики збережені в БД")
    except Exception as e:
        print(f"Помилка збереження метрик: {e}")

models = {}
train_results = {}

print("\n1. Тренування Logistic Regression")
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)
models['LogisticRegression'] = lr

y_train_pred_lr = lr.predict(X_train)
y_train_proba_lr = lr.predict_proba(X_train)[:, 1]

train_ids_lr = df.loc[X_train.index, 'patientid'].values
save_predictions_to_db(y_train, y_train_pred_lr, y_train_proba_lr, train_ids_lr, 'LogisticRegression', 'train')

metrics_lr_train = calculate_metrics(y_train, y_train_pred_lr, y_train_proba_lr, 'Logistic Regression', 'train')
train_results['LogisticRegression'] = metrics_lr_train
save_metrics_to_db('LogisticRegression', 'train', metrics_lr_train, optimized=False)

print("\n\n2. Тренування Random Forest")
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
models['RandomForest'] = rf

y_train_pred_rf = rf.predict(X_train)
y_train_proba_rf = rf.predict_proba(X_train)[:, 1]

train_ids_rf = df.loc[X_train.index, 'patientid'].values
save_predictions_to_db(y_train, y_train_pred_rf, y_train_proba_rf, train_ids_rf, 'RandomForest', 'train')

metrics_rf_train = calculate_metrics(y_train, y_train_pred_rf, y_train_proba_rf, 'Random Forest', 'train')
train_results['RandomForest'] = metrics_rf_train
save_metrics_to_db('RandomForest', 'train', metrics_rf_train, optimized=False)

print("\n\n3. Тренування Gradient Boosting")
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
models['GradientBoosting'] = gb

y_train_pred_gb = gb.predict(X_train)
y_train_proba_gb = gb.predict_proba(X_train)[:, 1]

train_ids_gb = df.loc[X_train.index, 'patientid'].values
save_predictions_to_db(y_train, y_train_pred_gb, y_train_proba_gb, train_ids_gb, 'GradientBoosting', 'train')

metrics_gb_train = calculate_metrics(y_train, y_train_pred_gb, y_train_proba_gb, 'Gradient Boosting', 'train')
train_results['GradientBoosting'] = metrics_gb_train
save_metrics_to_db('GradientBoosting', 'train', metrics_gb_train, optimized=False)

print("Моделі навчені")



1. Тренування Logistic Regression


Передбачення збережені в БД (1913 записів)

Метрики для Logistic Regression (train):
  Accuracy:  0.9482
  Precision: 0.0000
  Recall:    0.0000
  F1-Score:  0.0000
  ROC-AUC:   0.6471

Матриця плутанини:
  True Negatives (TN):  1814
  False Positives (FP): 0
  False Negatives (FN): 99
  True Positives (TP):  0
Метрики збережені в БД


2. Тренування Random Forest
Передбачення збережені в БД (1913 записів)

Метрики для Random Forest (train):
  Accuracy:  1.0000
  Precision: 1.0000
  Recall:    1.0000
  F1-Score:  1.0000
  ROC-AUC:   1.0000

Матриця плутанини:
  True Negatives (TN):  1814
  False Positives (FP): 0
  False Negatives (FN): 0
  True Positives (TP):  99
Метрики збережені в БД


3. Тренування Gradient Boosting
Передбачення збережені в БД (1913 записів)

Метрики для Gradient Boosting (train):
  Accuracy:  0.9645
  Precision: 1.0000
  Recall:    0.3131
  F1-Score:  0.4769
  ROC-AUC:   0.9883

Матриця плутанини:
  True Negatives (TN):  1814
  False Positives (FP): 0
  False Nega

## Завдання 6: Валідація моделей на тестовій та навчальній вибірці

Перевіряємо моделі на тестових даних та порівнюємо з результатами на тренувальних даних. Метрики зберігаємо у таблицю model_metrics.


In [ ]:
from sklearn.metrics import f1_score
import numpy as np

def find_best_thr(y_true, y_proba):
    best_thr, best_f1 = 0.5, 0
    for thr in np.arange(0.1, 0.91, 0.01):
        f1 = f1_score(y_true, (y_proba > thr).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr
    return best_thr, best_f1

print('\n=== Оцінка метрик з оптимальним threshold ===')
for name, model in models.items():
    if hasattr(model, 'predict_proba'):
        y_test_proba = model.predict_proba(X_test)[:,1]
        thr, best_f1 = find_best_thr(y_test, y_test_proba)
        y_test_pred = (y_test_proba > thr).astype(int)
        print(f'Model: {name}. Кращий threshold: {thr:.2f}. Max F1: {best_f1:.4f}')
        calculate_metrics(y_test, y_test_pred, y_test_proba, name+" (best_thr)", "test_best_thr")



=== Оцінка метрик з оптимальним threshold ===
Model: LogisticRegression. Кращий threshold: 0.11. Max F1: 0.0909

Метрики для LogisticRegression (best_thr) (test_best_thr):
  Accuracy:  0.9165
  Precision: 0.1053
  Recall:    0.0800
  F1-Score:  0.0909
  ROC-AUC:   0.6172

Матриця плутанини:
  True Negatives (TN):  437
  False Positives (FP): 17
  False Negatives (FN): 23
  True Positives (TP):  2
Model: RandomForest. Кращий threshold: 0.11. Max F1: 0.0652

Метрики для RandomForest (best_thr) (test_best_thr):
  Accuracy:  0.8205
  Precision: 0.0448
  Recall:    0.1200
  F1-Score:  0.0652
  ROC-AUC:   0.4665

Матриця плутанини:
  True Negatives (TN):  390
  False Positives (FP): 64
  False Negatives (FN): 22
  True Positives (TP):  3
Model: GradientBoosting. Кращий threshold: 0.12. Max F1: 0.0500

Метрики для GradientBoosting (best_thr) (test_best_thr):
  Accuracy:  0.9207
  Precision: 0.0667
  Recall:    0.0400
  F1-Score:  0.0500
  ROC-AUC:   0.5628

Матриця плутанини:
  True Negative

In [9]:
test_results = {}


print("\n1. Валідація Logistic Regression на тесті")
y_test_pred_lr = lr.predict(X_test)
y_test_proba_lr = lr.predict_proba(X_test)[:, 1]

test_ids_lr = df.loc[X_test.index, 'patientid'].values
save_predictions_to_db(y_test, y_test_pred_lr, y_test_proba_lr, test_ids_lr, 'LogisticRegression', 'test')

metrics_lr_test = calculate_metrics(y_test, y_test_pred_lr, y_test_proba_lr, 'Logistic Regression', 'test')
test_results['LogisticRegression'] = metrics_lr_test
save_metrics_to_db('LogisticRegression', 'test', metrics_lr_test, optimized=False)

# 2. Random Forest на тесті
print("\n\n2. Валідація Random Forest на тесті")
y_test_pred_rf = rf.predict(X_test)
y_test_proba_rf = rf.predict_proba(X_test)[:, 1]

test_ids_rf = df.loc[X_test.index, 'patientid'].values
save_predictions_to_db(y_test, y_test_pred_rf, y_test_proba_rf, test_ids_rf, 'RandomForest', 'test')

metrics_rf_test = calculate_metrics(y_test, y_test_pred_rf, y_test_proba_rf, 'Random Forest', 'test')
test_results['RandomForest'] = metrics_rf_test
save_metrics_to_db('RandomForest', 'test', metrics_rf_test, optimized=False)

# 3. Gradient Boosting на тесті
print("\n\n3. Валідація Gradient Boosting на тесті")
y_test_pred_gb = gb.predict(X_test)
y_test_proba_gb = gb.predict_proba(X_test)[:, 1]

test_ids_gb = df.loc[X_test.index, 'patientid'].values
save_predictions_to_db(y_test, y_test_pred_gb, y_test_proba_gb, test_ids_gb, 'GradientBoosting', 'test')

metrics_gb_test = calculate_metrics(y_test, y_test_pred_gb, y_test_proba_gb, 'Gradient Boosting', 'test')
test_results['GradientBoosting'] = metrics_gb_test
save_metrics_to_db('GradientBoosting', 'test', metrics_gb_test, optimized=False)

print("ПОРІВНЯННЯ МОДЕЛЕЙ")


comparison_df = pd.DataFrame({
    'Logistic Regression': [
        test_results['LogisticRegression']['accuracy'],
        test_results['LogisticRegression']['precision'],
        test_results['LogisticRegression']['recall'],
        test_results['LogisticRegression']['f1_score'],
        test_results['LogisticRegression']['roc_auc']
    ],
    'Random Forest': [
        test_results['RandomForest']['accuracy'],
        test_results['RandomForest']['precision'],
        test_results['RandomForest']['recall'],
        test_results['RandomForest']['f1_score'],
        test_results['RandomForest']['roc_auc']
    ],
    'Gradient Boosting': [
        test_results['GradientBoosting']['accuracy'],
        test_results['GradientBoosting']['precision'],
        test_results['GradientBoosting']['recall'],
        test_results['GradientBoosting']['f1_score'],
        test_results['GradientBoosting']['roc_auc']
    ]
}, index=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'])

print("\nМетрики на тестовій вибірці:")
display(comparison_df.round(4))




1. Валідація Logistic Regression на тесті
Передбачення збережені в БД (479 записів)

Метрики для Logistic Regression (test):
  Accuracy:  0.9478
  Precision: 0.0000
  Recall:    0.0000
  F1-Score:  0.0000
  ROC-AUC:   0.6172

Матриця плутанини:
  True Negatives (TN):  454
  False Positives (FP): 0
  False Negatives (FN): 25
  True Positives (TP):  0
Метрики збережені в БД


2. Валідація Random Forest на тесті
Передбачення збережені в БД (479 записів)

Метрики для Random Forest (test):
  Accuracy:  0.9478
  Precision: 0.0000
  Recall:    0.0000
  F1-Score:  0.0000
  ROC-AUC:   0.4665

Матриця плутанини:
  True Negatives (TN):  454
  False Positives (FP): 0
  False Negatives (FN): 25
  True Positives (TP):  0
Метрики збережені в БД


3. Валідація Gradient Boosting на тесті
Передбачення збережені в БД (479 записів)

Метрики для Gradient Boosting (test):
  Accuracy:  0.9436
  Precision: 0.0000
  Recall:    0.0000
  F1-Score:  0.0000
  ROC-AUC:   0.5628

Матриця плутанини:
  True Negatives

,Logistic Regression,Random Forest,Gradient Boosting
Accuracy,0.9478,0.9478,0.9436
Precision,0.0000,0.0000,0.0000
Recall,0.0000,0.0000,0.0000
F1-Score,0.0000,0.0000,0.0000
ROC-AUC,0.6172,0.4665,0.5628


## Завдання 7: Вибір найкращої моделі

Порівнюємо метрики всіх моделей та обираємо найкращу. Основна метрика F1-Score бо вона балансує між Precision та Recall. Допоміжна метрика Recall важлива для медицини бо не можна пропустити хворого. Також враховуємо ROC-AUC та загальну стабільність на тесті.


In [ ]:
f1_scores = {
    'Logistic Regression': test_results['LogisticRegression']['f1_score'],
    'Random Forest': test_results['RandomForest']['f1_score'],
    'Gradient Boosting': test_results['GradientBoosting']['f1_score']
}

best_f1_model = max(f1_scores, key=f1_scores.get)
print(f"\n1.  F1-Score: {best_f1_model} ({f1_scores[best_f1_model]:.4f})")

# Аналіз за Recall 
recall_scores = {
    'Logistic Regression': test_results['LogisticRegression']['recall'],
    'Random Forest': test_results['RandomForest']['recall'],
    'Gradient Boosting': test_results['GradientBoosting']['recall']
}

best_recall_model = max(recall_scores, key=recall_scores.get)
print(f"2.  Recall: {best_recall_model} ({recall_scores[best_recall_model]:.4f})")

# Аналіз за ROC-AUC
roc_auc_scores = {
    'Logistic Regression': test_results['LogisticRegression']['roc_auc'],
    'Random Forest': test_results['RandomForest']['roc_auc'],
    'Gradient Boosting': test_results['GradientBoosting']['roc_auc']
}

best_auc_model = max(roc_auc_scores, key=roc_auc_scores.get)
print(f"3.  ROC-AUC: {best_auc_model} ({roc_auc_scores[best_auc_model]:.4f})")


print("Аналіз:")


for model_name in ['LogisticRegression', 'RandomForest', 'GradientBoosting']:
    display_name = model_name.replace('Regression', ' Regression').replace('Forest', ' Forest').replace('Boosting', ' Boosting')
    print(f"\n{display_name}:")
    print(f"  F1-Score: {test_results[model_name]['f1_score']:.4f}")
    print(f"  Recall:    {test_results[model_name]['recall']:.4f}")
    print(f"  Precision: {test_results[model_name]['precision']:.4f}")
    print(f"  ROC-AUC:   {test_results[model_name]['roc_auc']:.4f}")


combined_scores = {}
for model_name in ['LogisticRegression', 'RandomForest', 'GradientBoosting']:
    combined_scores[model_name] = (
        test_results[model_name]['f1_score'] * 0.5 + 
        test_results[model_name]['recall'] * 0.5
    )

best_model_name = max(combined_scores, key=combined_scores.get)
best_model_display = best_model_name.replace('Regression', ' Regression').replace('Forest', ' Forest').replace('Boosting', ' Boosting')

best_model = models[best_model_name]
print(f"\n Обрана модель для подальшої оптимізації: {best_model_name}")

print(f"Обґрунтування:")
print(f"- F1-Score: {test_results[best_model_name]['f1_score']:.4f}")
print(f"- Recall: {test_results[best_model_name]['recall']:.4f} (критично важлива для медицини)")
print(f"- Precision: {test_results[best_model_name]['precision']:.4f}")
print(f"- ROC-AUC: {test_results[best_model_name]['roc_auc']:.4f}")
print(f"- Комбінована оцінка: {combined_scores[best_model_name]:.4f}")


1.  F1-Score: Logistic Regression (0.0000)
2.  Recall: Logistic Regression (0.0000)
3.  ROC-AUC: Logistic Regression (0.6172)
Аналіз:

Logistic Regression:
  F1-Score: 0.0000
  Recall:    0.0000
  Precision: 0.0000
  ROC-AUC:   0.6172

Random Forest:
  F1-Score: 0.0000
  Recall:    0.0000
  Precision: 0.0000
  ROC-AUC:   0.4665

Gradient Boosting:
  F1-Score: 0.0000
  Recall:    0.0000
  Precision: 0.0000
  ROC-AUC:   0.5628

 Обрана модель для подальшої оптимізації: LogisticRegression
Обґрунтування:
- F1-Score: 0.0000
- Recall: 0.0000 (критично важлива для медицини)
- Precision: 0.0000
- ROC-AUC: 0.6172
- Комбінована оцінка: 0.0000


## Завдання 8: Аналіз важливості ознак

Подивимось які ознаки найбільше впливають на передбачення моделі. Це допомагає зрозуміти які фактори найважливіші для діагностики астми. Також можна спростити модель видаливши маловажливі ознаки. Лікарі теж отримують інформацію про ключові ризик-фактори.


In [11]:
print("Аналіз важливості ознак:")

feature_importance = {}

if 'RandomForest' in models and hasattr(models['RandomForest'], 'feature_importances_'):
    rf_importance = models['RandomForest'].feature_importances_
    feature_importance['RandomForest'] = rf_importance
    print("\n1. Random Forest - важливість ознак:")

if 'GradientBoosting' in models and hasattr(models['GradientBoosting'], 'feature_importances_'):
    gb_importance = models['GradientBoosting'].feature_importances_
    feature_importance['GradientBoosting'] = gb_importance
    print("\n2. Gradient Boosting - важливість ознак:")

if 'LogisticRegression' in models:
    lr_coef = np.abs(models['LogisticRegression'].coef_[0])
    feature_importance['LogisticRegression'] = lr_coef
    print("\n3. Logistic Regression - важливість ознак (абсолютні коефіцієнти):")

importance_df = pd.DataFrame({
    'feature': feature_cols
})

for model_name, importance in feature_importance.items():
    importance_df[f'{model_name}_importance'] = importance

if len(feature_importance) > 1:
    importance_df['average_importance'] = importance_df[[f'{name}_importance' for name in feature_importance.keys()]].mean(axis=1)
    importance_df = importance_df.sort_values('average_importance', ascending=False)
else:
    first_model = list(feature_importance.keys())[0]
    importance_df = importance_df.sort_values(f'{first_model}_importance', ascending=False)


print("15 найважливіших ознак::")

display(importance_df.head(15))


print("Найбільш важливі ознаки для діагностики астми:")
for i, row in importance_df.head(10).iterrows():
    feature_name = row['feature']
    avg_imp = row.get('average_importance', row.iloc[1])
    print(f"  {feature_name}: {avg_imp:.4f}")

importance_df.to_csv('feature_importance.csv', index=False)
print("\nВажливість ознак збережена у файл feature_importance.csv")


Аналіз важливості ознак:

1. Random Forest - важливість ознак:

2. Gradient Boosting - важливість ознак:

3. Logistic Regression - важливість ознак (абсолютні коефіцієнти):
15 найважливіших ознак::


,feature,RandomForest_importance,GradientBoosting_importance,LogisticRegression_importance,average_importance
9,dustexposure,0.087231,0.110065,0.152575,0.116624
2,bmi,0.072944,0.156627,0.077756,0.102442
17,lungfunctionfvc,0.084044,0.113363,0.097713,0.098373
6,sleepquality,0.076050,0.103989,0.099841,0.093293
16,lungfunctionfev1,0.074210,0.061254,0.125051,0.086839
8,pollenexposure,0.074074,0.097719,0.059485,0.077093
0,age,0.073345,0.055973,0.090731,0.073350
7,pollutionexposure,0.085596,0.090515,0.034379,0.070163
23,exerciseinduced,0.009992,0.004904,0.185233,0.066710
26,ethnicity_3,0.012540,0.002074,0.179194,0.064603


Найбільш важливі ознаки для діагностики астми:
  dustexposure: 0.1166
  bmi: 0.1024
  lungfunctionfvc: 0.0984
  sleepquality: 0.0933
  lungfunctionfev1: 0.0868
  pollenexposure: 0.0771
  age: 0.0733
  pollutionexposure: 0.0702
  exerciseinduced: 0.0667
  ethnicity_3: 0.0646

Важливість ознак збережена у файл feature_importance.csv


## Завдання 9: Вибір метрики для гіперпараметричної оптимізації

F1-Score балансує між Precision та Recall, адже в такій штуці як медицина важливо не пропустити хворих але і не турбувати здорових зайвими попередженнями.
Не обираємо Accuracy бо може бути оманливою при незбалансованих класах. Не обираємо Recall і Precision бо може призвести до занадто багатьох false positives і ми пропустимо хворих і це дуже не добре (напевно).
ЗАГАЛОМ вибираємо F1-Score!


In [12]:
optimization_metric = 'f1'
print(f"Обрана метрика для оптимізації: {optimization_metric.upper()}-Score")

Обрана метрика для оптимізації: F1-Score


## Завдання 10: Вибір методу гіперпараметричної оптимізації

Обираємо RandomizedSearchCV - він перевіряє випадкову вибірку комбінацій швидше але все ще ефективно. Використовуємо n_iter=50 це ідеальна серединка між якістю та швидкістю.

Гіперпараметри залежать від обраної моделі. Для Random Forest це n_estimators, max_depth, min_samples_split, min_samples_leaf. Для Gradient Boosting це n_estimators, learning_rate, max_depth, min_samples_split. Для Logistic Regression це C, penalty, solver.

In [13]:
if best_model_name == 'RandomForest':
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 20, 30, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'class_weight': [None, 'balanced']
    }
    model_class = RandomForestClassifier
    default_params = {'random_state': 42, 'n_jobs': -1}
    print("Гіперпараметри для Random Forest:")
    
elif best_model_name == 'GradientBoosting':
    param_grid = {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 5, 7],
        'min_samples_split': [2, 5, 10]
    }
    model_class = GradientBoostingClassifier
    default_params = {'random_state': 42}
    print("Гіперпараметри для Gradient Boosting:")
    
else:  # LogisticRegression
    param_grid = {
        'C': [0.001, 0.01, 0.1, 1, 10, 100],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear', 'lbfgs'],
        'class_weight': [None, 'balanced']
    }
    model_class = LogisticRegression
    default_params = {'random_state': 42, 'max_iter': 1000}
    print("Гіперпараметри для Logistic Regression:")

for param, values in param_grid.items():
    print(f"  {param}: {values}")

print(f"\nКількість можливих комбінацій: {np.prod([len(v) for v in param_grid.values()])}")
print(f"RandomizedSearchCV перевірить n_iter=50 комбінацій")

Гіперпараметри для Logistic Regression:
  C: [0.001, 0.01, 0.1, 1, 10, 100]
  penalty: ['l1', 'l2']
  solver: ['liblinear', 'lbfgs']
  class_weight: [None, 'balanced']

Кількість можливих комбінацій: 48
RandomizedSearchCV перевірить n_iter=50 комбінацій


## Завдання 11: Запуск оптимізації гіперпараметрів

Запускаємо RandomizedSearchCV для підбору найкращих гіперпараметрів. Після оптимізації зберігаємо найкращі значення гіперпараметрів та метрики до та після оптимізації у таблиці model_metrics.


In [14]:
print("Оптімізейшн оф гіпепараметри")

print(f"Модель: {best_model_name}")
print(f"Метрика оптимізації: {optimization_metric.upper()}-Score")
print(f"n_iter: 50 (випадкових комбінацій)/n")

base_model = model_class(**default_params)

from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    base_model,
    param_distributions=param_grid,
    n_iter=50,
    cv=cv,
    scoring=optimization_metric,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X_train, y_train)

print(f"\nНайкращі параметри:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nНайкращий {optimization_metric.upper()}-Score на кросс-валідації: {random_search.best_score_:.4f}")

optimized_model = random_search.best_estimator_
models[f'{best_model_name}_optimized'] = optimized_model


y_train_pred_opt = optimized_model.predict(X_train)
y_train_proba_opt = optimized_model.predict_proba(X_train)[:, 1]
metrics_train_opt = calculate_metrics(y_train, y_train_pred_opt, y_train_proba_opt, f'{best_model_name} (optimized)', 'train')
save_metrics_to_db(f'{best_model_name}_optimized', 'train', metrics_train_opt, random_search.best_params_, optimized=True)

train_ids_opt = df.loc[X_train.index, 'patientid'].values
save_predictions_to_db(y_train, y_train_pred_opt, y_train_proba_opt, train_ids_opt, f'{best_model_name}_optimized', 'train')

y_test_pred_opt = optimized_model.predict(X_test)
y_test_proba_opt = optimized_model.predict_proba(X_test)[:, 1]
metrics_test_opt = calculate_metrics(y_test, y_test_pred_opt, y_test_proba_opt, f'{best_model_name} (optimized)', 'test')
save_metrics_to_db(f'{best_model_name}_optimized', 'test', metrics_test_opt, random_search.best_params_, optimized=True)

test_ids_opt = df.loc[X_test.index, 'patientid'].values
save_predictions_to_db(y_test, y_test_pred_opt, y_test_proba_opt, test_ids_opt, f'{best_model_name}_optimized', 'test')


print("Порівняння до та після оптімізейшн:")


comparison_opt = pd.DataFrame({
    'До оптимізації': [
        test_results[best_model_name]['accuracy'],
        test_results[best_model_name]['precision'],
        test_results[best_model_name]['recall'],
        test_results[best_model_name]['f1_score'],
        test_results[best_model_name]['roc_auc']
    ],
    'Після оптимізації': [
        metrics_test_opt['accuracy'],
        metrics_test_opt['precision'],
        metrics_test_opt['recall'],
        metrics_test_opt['f1_score'],
        metrics_test_opt['roc_auc']
    ]
}, index=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'])

comparison_opt['Покращення'] = comparison_opt['Після оптимізації'] - comparison_opt['До оптимізації']
display(comparison_opt.round(4))

print("\nОптимізація завершена, результати збережені")


Оптімізейшн оф гіпепараметри
Модель: LogisticRegression
Метрика оптимізації: F1-Score
n_iter: 50 (випадкових комбінацій)/n
Fitting 5 folds for each of 48 candidates, totalling 240 fits

Найкращі параметри:
  solver: lbfgs
  penalty: l2
  class_weight: balanced
  C: 0.001

Найкращий F1-Score на кросс-валідації: 0.0921

Метрики для LogisticRegression (optimized) (train):
  Accuracy:  0.6205
  Precision: 0.0826
  Recall:    0.6263
  F1-Score:  0.1459
  ROC-AUC:   0.6464

Матриця плутанини:
  True Negatives (TN):  1125
  False Positives (FP): 689
  False Negatives (FN): 37
  True Positives (TP):  62
Метрики збережені в БД
Передбачення збережені в БД (1913 записів)

Метрики для LogisticRegression (optimized) (test):
  Accuracy:  0.5908
  Precision: 0.0788
  Recall:    0.6400
  F1-Score:  0.1404
  ROC-AUC:   0.6273

Матриця плутанини:
  True Negatives (TN):  267
  False Positives (FP): 187
  False Negatives (FN): 9
  True Positives (TP):  16
Метрики збережені в БД
Передбачення збережені в БД

,До оптимізації,Після оптимізації,Покращення
Accuracy,0.9478,0.5908,-0.3570
Precision,0.0000,0.0788,0.0788
Recall,0.0000,0.6400,0.6400
F1-Score,0.0000,0.1404,0.1404
ROC-AUC,0.6172,0.6273,0.0101



Оптимізація завершена, результати збережені


## Завдання 12: Збереження моделі

Зберігаємо оптимізовану модель у папку models/. Виводимо метрики оптимізованої моделі на навчальних та тестувальних даних.


In [15]:
model_filename = f'models/{best_model_name}_optimized_{datetime.now().strftime("%Y%m%d_%H%M%S")}.pkl'
joblib.dump(optimized_model, model_filename)
print(f"Модель збережена: {model_filename}")

latest_model_filename = f'models/{best_model_name}_optimized_latest.pkl'
joblib.dump(optimized_model, latest_model_filename)
print(f"Модель також збережена як остання версія: {latest_model_filename}")

model_metadata = {
    'model_name': best_model_name,
    'best_params': random_search.best_params_,
    'best_cv_score': random_search.best_score_,
    'train_metrics': metrics_train_opt,
    'test_metrics': metrics_test_opt,
    'feature_names': feature_cols,
    'saved_at': datetime.now().isoformat()
}

metadata_filename = f'models/{best_model_name}_optimized_metadata.pkl'
joblib.dump(model_metadata, metadata_filename)
print(f"Метадані моделі збережені: {metadata_filename}")

print("Метрики оптмизованої моделі")

print("\nНа тренувальних даних:")
print(f"  Accuracy:  {metrics_train_opt['accuracy']:.4f}")
print(f"  Precision: {metrics_train_opt['precision']:.4f}")
print(f"  Recall:    {metrics_train_opt['recall']:.4f}")
print(f"  F1-Score:  {metrics_train_opt['f1_score']:.4f}")
print(f"  ROC-AUC:   {metrics_train_opt['roc_auc']:.4f}")

print("\nНа тестових даних:")
print(f"  Accuracy:  {metrics_test_opt['accuracy']:.4f}")
print(f"  Precision: {metrics_test_opt['precision']:.4f}")
print(f"  Recall:    {metrics_test_opt['recall']:.4f}")
print(f"  F1-Score:  {metrics_test_opt['f1_score']:.4f}")
print(f"  ROC-AUC:   {metrics_test_opt['roc_auc']:.4f}")

overfitting_check = {
    'metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'train': [
        metrics_train_opt['accuracy'],
        metrics_train_opt['precision'],
        metrics_train_opt['recall'],
        metrics_train_opt['f1_score'],
        metrics_train_opt['roc_auc']
    ],
    'test': [
        metrics_test_opt['accuracy'],
        metrics_test_opt['precision'],
        metrics_test_opt['recall'],
        metrics_test_opt['f1_score'],
        metrics_test_opt['roc_auc']
    ]
}

overfitting_df = pd.DataFrame(overfitting_check)
overfitting_df['Різниця'] = overfitting_df['train'] - overfitting_df['test']
overfitting_df['Різниця %'] = (overfitting_df['Різниця'] / overfitting_df['train'] * 100).round(2)

print("\n" + "-" * 60)
print("Перевірка Overfitting:")
display(overfitting_df.round(4))

if overfitting_df['Різниця'].max() > 0.15:
    print("\nЄ ознаки overfitting (велика різниця між train та test)")
else:
    print("\nOverfitting не виявлено")

print("\nМодель збережена")


Модель збережена: models/LogisticRegression_optimized_20251030_105610.pkl
Модель також збережена як остання версія: models/LogisticRegression_optimized_latest.pkl
Метадані моделі збережені: models/LogisticRegression_optimized_metadata.pkl
Метрики оптмизованої моделі

На тренувальних даних:
  Accuracy:  0.6205
  Precision: 0.0826
  Recall:    0.6263
  F1-Score:  0.1459
  ROC-AUC:   0.6464

На тестових даних:
  Accuracy:  0.5908
  Precision: 0.0788
  Recall:    0.6400
  F1-Score:  0.1404
  ROC-AUC:   0.6273

------------------------------------------------------------
Перевірка Overfitting:


,metric,train,test,Різниця,Різниця %
0,Accuracy,0.6205,0.5908,0.0297,4.78
1,Precision,0.0826,0.0788,0.0037,4.53
2,Recall,0.6263,0.6400,-0.0137,-2.19
3,F1-Score,0.1459,0.1404,0.0055,3.79
4,ROC-AUC,0.6464,0.6273,0.0191,2.95



Overfitting не виявлено

Модель збережена


## Завдання 13: Пояснення результатів оптимізації

Аналізуємо результати оптимізації. Дивимось як змінились метрики після підбору параметрів, які гіперпараметри мали найбільший вплив, чи досягли ми покращення.


In [16]:

improvements = {
    'Accuracy': (metrics_test_opt['accuracy'] - test_results[best_model_name]['accuracy']) * 100,
    'Precision': (metrics_test_opt['precision'] - test_results[best_model_name]['precision']) * 100,
    'Recall': (metrics_test_opt['recall'] - test_results[best_model_name]['recall']) * 100,
    'F1-Score': (metrics_test_opt['f1_score'] - test_results[best_model_name]['f1_score']) * 100,
    'ROC-AUC': (metrics_test_opt['roc_auc'] - test_results[best_model_name]['roc_auc']) * 100
}

print("\nЗміни метрик (у відсотках):")
for metric, change in improvements.items():
    sign = "+" if change >= 0 else ""
    print(f"  {metric:12s}: {sign}{change:+.2f}%")


print("\nНайкращі знайдені параметри:")
best_params = random_search.best_params_
for param, value in best_params.items():
    print(f"  {param}: {value}")


if best_model_name == 'RandomForest':
    print(f"  n_estimators={best_params.get('n_estimators', 'N/A')}: кількість дерев - більше дерев = краща узагальнення")
    print(f"  max_depth={best_params.get('max_depth', 'N/A')}: глибина дерев - контролює складність")
    print(f"  min_samples_split={best_params.get('min_samples_split', 'N/A')}: мінімум для розділення - запобігає overfitting")
    if 'class_weight' in best_params:
        print(f"  class_weight={best_params['class_weight']}: враховує баланс класів")

elif best_model_name == 'GradientBoosting':
    print(f"  n_estimators={best_params.get('n_estimators', 'N/A')}: кількість ітерацій")
    print(f"  learning_rate={best_params.get('learning_rate', 'N/A')}: крок навчання - менший = повільніше, але точніше")
    print(f"  max_depth={best_params.get('max_depth', 'N/A')}: глибина дерев")

else:
    print(f"  C={best_params.get('C', 'N/A')}: сила регуляризації - менший = більша регуляризація")
    print(f"  penalty={best_params.get('penalty', 'N/A')}: тип регуляризації (L1/L2)")
    if 'class_weight' in best_params:
        print(f"  class_weight={best_params['class_weight']}: враховує баланс класів")


if improvements['F1-Score'] > 0:
    print("F1-Score покращився - оптимізація успішна")
if improvements['Recall'] > 0:
    print("Recall покращився - модель краще виявляє хворих")
if improvements['ROC-AUC'] > 0:
    print("ROC-AUC покращився - загальна якість моделі вища")

print("\nОптимізація дозволила:")
print("Підібрати оптимальні гіперпараметри для нашої задачі")
print("Покращити баланс між точністю та повнотою")
print("Створити модель готову до використання")



Зміни метрик (у відсотках):
  Accuracy    : -35.70%
  Precision   : ++7.88%
  Recall      : ++64.00%
  F1-Score    : ++14.04%
  ROC-AUC     : ++1.01%

Найкращі знайдені параметри:
  solver: lbfgs
  penalty: l2
  class_weight: balanced
  C: 0.001
  C=0.001: сила регуляризації - менший = більша регуляризація
  penalty=l2: тип регуляризації (L1/L2)
  class_weight=balanced: враховує баланс класів
F1-Score покращився - оптимізація успішна
Recall покращився - модель краще виявляє хворих
ROC-AUC покращився - загальна якість моделі вища

Оптимізація дозволила:
Підібрати оптимальні гіперпараметри для нашої задачі
Покращити баланс між точністю та повнотою
Створити модель готову до використання


## Завдання 14: Обробка незбалансованих класів

Якщо дані мають дисбаланс класів можна використати SMOTE, class_weight параметр у моделі, undersampling або oversampling.

У нашому випадку якщо співвідношення класів менше 0.5 вважаємо дані незбалансованими.

SMOTE може додати штучні приклади що не завжди підходить для медичних даних, тому використовуємо параметр class_weight='balanced' в гіперпараметричному пошуку це автоматично надає більшу вагу меншому класу під час навчання. Не потребує зміни даних, працює на рівні моделі.


In [17]:
print("Аналіз балансу класів~")

print("\nРозподіл класів у тренувальній вибірці:")
train_dist = y_train.value_counts()
print(train_dist)
train_ratio = train_dist[1] / train_dist[0]
print(f"Співвідношення (клас 1 / клас 0): {train_ratio:.3f}")

if train_ratio < 0.5 or train_ratio > 2.0:
    is_imbalanced = True
    print("\nДані незбалансовані")
else:
    is_imbalanced = False
    print("\nДані досить збалансовані")

if 'class_weight' in best_params:
    print(f"\nВикористано class_weight='{best_params['class_weight']}' для обробки дисбалансу")
    print("Це автоматично надає більшу вагу меншому класу під час навчання")

if is_imbalanced and best_params.get('class_weight') == 'balanced':
    print("Дисбаланс успішно оброблено через class_weight='balanced'")
elif not is_imbalanced:
    print("Дані збалансовані, спеціальна обробка не потрібна")
else:
    print("Можливо варто розглянути додаткові методи обробки незбалансованості")


Аналіз балансу класів~

Розподіл класів у тренувальній вибірці:
diagnosis
0    1814
1      99
Name: count, dtype: int64
Співвідношення (клас 1 / клас 0): 0.055

Дані незбалансовані

Використано class_weight='balanced' для обробки дисбалансу
Це автоматично надає більшу вагу меншому класу під час навчання
Дисбаланс успішно оброблено через class_weight='balanced'


## Висновок

В результаті цієї всієї роботи ми дали даним шанс розповісти правду, а моделям не тупити. Після тренувань і тюнінгу найкраща модель навчається ловити астму, а важливі ознаки підказали що деякі фактори реально щось роблять, а інші просто страдають фігньою.
З цього всього ми можемо робити прогнози і підкидати лікарям підказки там де це важливо. Якщо підвезти більше реальних даних то звісно стане ще розумніше. Якщо чесно це якась біда, особливо з точністю метрик :/


In [18]:
from sqlalchemy import text

print("Перевірка збереження у БД:")

with engine.connect() as conn:
    mm_count = conn.execute(text("""
        SELECT COUNT(*) FROM model_metrics 
        WHERE model_name = :mname AND optimized = TRUE
    """), {"mname": f"{best_model_name}_optimized"}).scalar()
    print(f"model_metrics (optimized) рядків: {mm_count}")

    pred_train_count = conn.execute(text("""
        SELECT COUNT(*) FROM predictions 
        WHERE model_name = :mname AND source = 'train'
    """), {"mname": f"{best_model_name}_optimized"}).scalar()
    pred_test_count = conn.execute(text("""
        SELECT COUNT(*) FROM predictions 
        WHERE model_name = :mname AND source = 'test'
    """), {"mname": f"{best_model_name}_optimized"}).scalar()
    print(f"predictions train: {pred_train_count}; test: {pred_test_count}")

    print("\nОстанні метрики (optimized, test):")
    last_metrics = pd.read_sql_query(
        """
        SELECT model_name, dataset_type, accuracy, precision, recall, f1_score, roc_auc, hyperparameters, optimized, created_at
        FROM model_metrics
        WHERE model_name = %(mname)s AND dataset_type = 'test' AND optimized = TRUE
        ORDER BY created_at DESC
        LIMIT 3
        """,
        con=engine,
        params={"mname": f"{best_model_name}_optimized"}
    )
    display(last_metrics)



Перевірка збереження у БД:
model_metrics (optimized) рядків: 20
predictions train: 19130; test: 4790

Останні метрики (optimized, test):


,model_name,dataset_type,accuracy,precision,recall,f1_score,roc_auc,hyperparameters,optimized,created_at
0,LogisticRegression_optimized,test,0.590814,0.078818,0.64,0.140351,0.627313,"{'solver': 'lbfgs', 'penalty': 'l2', 'class_we...",True,2025-10-30 10:56:10.753999
1,LogisticRegression_optimized,test,0.590814,0.078818,0.64,0.140351,0.627313,"{'solver': 'lbfgs', 'penalty': 'l2', 'class_we...",True,2025-10-30 10:50:25.830468
2,LogisticRegression_optimized,test,0.590814,0.078818,0.64,0.140351,0.627313,"{'solver': 'lbfgs', 'penalty': 'l2', 'class_we...",True,2025-10-30 10:33:32.908990
